In [ ]:
from cemp_software_settings import load_and_apply_settings
result = load_and_apply_settings()
print(result)

In [ ]:
import os
import pandas as pd
import re

In [ ]:

try:
    from openpyxl import load_workbook
except ImportError:
    print("Public message removed for release.")
    exit()

In [ ]:

def get_filename_without_extension(xlsx_path):
    
    
    base_name = os.path.basename(xlsx_path)
    
    file_name_without_extension = os.path.splitext(base_name)[0]
    return file_name_without_extension

In [ ]:

def extract_energy_from_out(file_path):
    energy = None
    with open(file_path, 'r') as file:
        lines = file.readlines()
        energy_line = ''
        for i, line in enumerate(lines):
            
            line = line.strip().replace('\n', '')
            
            if line.strip().endswith('H') and i+1 < len(lines):
                next_line = lines[i+1]
                if next_line.strip().startswith('F='):
                    
                    line = line.strip() + next_line.strip()
            
            if line.strip().endswith('HF') and i+1 < len(lines):
                next_line = lines[i+1]
                if next_line.strip().startswith('='):
                    
                    line = line.strip() + next_line.strip()
            
            if 'HF=' in line:
                
                start = line.find('HF=') + 3  
                energy_line = line[start:]
                
                if '\\' in energy_line:
                    energy = energy_line.split('\\')[0].strip()
                    return energy
                else:
                    
                    for j in range(i+1, len(lines)):
                        energy_line += lines[j].strip()
                        if '\\' in lines[j]:
                            energy = energy_line.split('\\')[0].strip()
                            return energy
    return energy  

In [ ]:
def extract_dipole_moment(file_path):
    
    try:
        with open(file_path, 'r') as file:
            lines = file.readlines()  

        
        dipole_moment_lines = [
            i for i, line in enumerate(lines)
            if re.search(r"Dipole moment \(field-independent basis, Debye\):", line)
        ]

        
        if not dipole_moment_lines:
            print("Public message removed for release.")
            return None

        
        last_line_number = dipole_moment_lines[-1]

        
        next_line = lines[last_line_number + 1]
        match = re.search(r"Tot=\s*(-?\d+\.\d+)", next_line)

        
        if match:
            return float(match.group(1))
        else:
            print("Public message removed for release.")
            return None

    except FileNotFoundError:
        print("Public message removed for release.")
        return None
    except Exception as e:
        print("Public message removed for release.")
        return None

In [ ]:

def extract_homo_lumo(file_path):
    
    try:
        filename = get_filename_without_extension(file_path)
        
        with open(file_path, 'r') as file:
            content = file.readlines()
        
        
        pop_analysis_lines = [i for i, line in enumerate(content) if re.match(r'\s*Population analysis', line)]
        
        
        if len(pop_analysis_lines) == 0:
            raise ValueError("Public message removed for release.")

        
        last_population_analysis = pop_analysis_lines[-1]

        
        homo, lumo = None, None
        for line_number in range(last_population_analysis, len(content)):
            occ_match = re.match(r'\s*Alpha\s+occ.\s+eigenvalues\s+--(.*)', content[line_number])
            virt_match = re.match(r'\s*Alpha\s+virt.\s+eigenvalues\s+--(.*)', content[line_number + 1]) if line_number + 1 < len(content) else None
            
            if occ_match and virt_match:
                
                homo = float(occ_match.group(1).strip().split()[-1])  
                lumo = float(virt_match.group(1).strip().split()[0])  
                break
        
        
        if homo is None or lumo is None:
            raise ValueError("Public message removed for release.")
        
        
        return homo, lumo

    except FileNotFoundError:
        
        raise FileNotFoundError("Public message removed for release.")
    except Exception as e:
        
        raise Exception("Public message removed for release.")

In [ ]:

def extract_entropy(file_path):
    
    entropy_pattern = re.compile(r'\s*Total\s+[0-9.-]+\s+[0-9.-]+\s+([0-9.-]+)')
    try:
        with open(file_path, 'r') as file:
            file_content = file.read()  
            match = entropy_pattern.search(file_content)
            if match:
                
                entropy_value_cal = float(match.group(1))
                
                entropy_value_joules = entropy_value_cal * 4.184 
                return entropy_value_joules
    except FileNotFoundError:
        print(f"The file {file_path} was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

    return None  

In [ ]:

def extract_enthalpy_correction(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            if "Thermal correction to Enthalpy" in line:
                
                match = re.search(r"=\s*([-\d.]+)", line)
                if match:
                    return float(match.group(1))
    return None

In [ ]:

def extract_gibbs_correction(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            if "Thermal correction to Gibbs Free Energy" in line:
                
                match = re.search(r"=\s*([-\d.]+)", line)
                if match:
                    return float(match.group(1))
    return None

In [ ]:


def extract_properties_from_energycal(excel_file, success_dir):
    
    df = pd.read_excel(excel_file)
    
    
    if 'Dimer Energy (Hatree)' not in df.columns:
        
        df['Dimer Energy (Hatree)'] = 0.0
    
    
    for filename in os.listdir(success_dir):
        if filename.endswith('.out'):
            
            dimer_name = filename.rsplit('.', 1)[0]
            
            energy = extract_energy_from_out(os.path.join(success_dir, filename))
            
            
            energy = float(energy) if energy else 0.0
            
            
            
            df.loc[df['Dimer Name'] == dimer_name, 'Dimer Energy (Hatree)'] = energy

    
    
    df.to_excel(excel_file, index=False)

In [ ]:

def extract_gibbs_correction(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            if "Thermal correction to Gibbs Free Energy" in line:
                
                match = re.search(r"=\s*([-\d.]+)", line)
                if match:
                    return float(match.group(1))
    return None

In [ ]:

def extract_properties_from_opt_freq(excel_file, success_dir):
    
    
    df = pd.read_excel(excel_file)
    
    
    df["Dimer Thermal correction to Gibbs Free Energy (Hatree)"] = 0.0 
    df["Dimer Thermal correction to Enthalpy (Hatree)"] = 0.0 
    df['Dimer Entropy (J/mol·K)'] = 0.0 
    df['Dimer HOMO (Hatree)'] = 0.0 
    df['Dimer LOMO (Hatree)'] = 0.0 
    df['Dimer Dipole (Debye)'] = 0.0 
    
    
    for filename in os.listdir(success_dir):
        if filename.endswith('.out'):
            
            dimer_name = filename.rsplit('.', 1)[0]
            
            gibbs_correction = extract_gibbs_correction(os.path.join(success_dir, filename)) 
            enthalpy_correction = extract_enthalpy_correction(os.path.join(success_dir, filename)) 
            entropy = extract_entropy(os.path.join(success_dir, filename)) 
            homo, lumo = extract_homo_lumo(os.path.join(success_dir, filename)) 
            dipole = extract_dipole_moment(os.path.join(success_dir, filename))
            
            
            dipole = float(dipole) if dipole else 0.0
            
            
            
            
            df.loc[df['Dimer Name'] == dimer_name, 'Dimer Thermal correction to Gibbs Free Energy (Hatree)'] = gibbs_correction
            df.loc[df['Dimer Name'] == dimer_name, 'Dimer Thermal correction to Enthalpy (Hatree)'] = enthalpy_correction
            df.loc[df['Dimer Name'] == dimer_name, 'Dimer Entropy (J/mol·K)'] = entropy
            df.loc[df['Dimer Name'] == dimer_name, 'Dimer HOMO (Hatree)'] = homo
            df.loc[df['Dimer Name'] == dimer_name, 'Dimer LUMO (Hatree)'] = lumo
            df.loc[df['Dimer Name'] == dimer_name, 'Dimer Dipole (Debye)'] = dipole

                   
    
    df.to_excel(excel_file, index=False)

In [ ]:

def data_processing(excel_file):
    
    df = pd.read_excel(excel_file)
    
    df["Dimer Gibbs Free Energy (Hatree)"] = df['Dimer Energy (Hatree)'] + df["Dimer Thermal correction to Gibbs Free Energy (Hatree)"] 
    df["Dimer Enthalpy (Hatree)"] = df['Dimer Energy (Hatree)'] + df["Dimer Thermal correction to Enthalpy (Hatree)"] 
    df["Binding energy (kJ/mol)"] = (df["Dimer Energy (Hatree)"] - df["Component A Energy (Hatree)"] - df["Component B Energy (Hatree)"])*2625.5
    
    
    df.to_excel(excel_file, index=False)

In [ ]:

component_excel_path = 'Dimer.xlsx'
opt_success_dir = "dimer_gas/opt+freq/success"
energy_success_dir = "dimer_gas/energy/success"


extract_properties_from_energycal(component_excel_path, energy_success_dir)
extract_properties_from_opt_freq(component_excel_path, opt_success_dir)
data_processing(component_excel_path)

In [ ]:


def remove_hash_from_names(df):
    
    
    columns_to_process = ['Dimer Name', 'Component Name A', 'Component Name B']
    
    for col in columns_to_process:
        if col in df.columns:
            
            
            df[col] = df[col].apply(lambda x: x[:-5] if isinstance(x, str) and len(x) >= 5 else x)
    
    return df
df = pd.read_excel(component_excel_path)
df = remove_hash_from_names(df)
df.to_excel(component_excel_path)